# Ferrero demo – load the case-intelligence lakehouse

Loads the synthetic demo data pack into Delta tables in the attached Lakehouse, adds a date dimension and the derived columns the dashboards use, and checks referential integrity.

**Before you run**
1. Create a Lakehouse (e.g. `lh_ferrero_demo`) in the demo workspace on the F4 capacity and attach it to this notebook as the default lakehouse.
2. Upload the `csv/` folder from the data pack to `Files/ferrero_demo/csv` (Lakehouse explorer → Files → Upload folder).
3. Run all cells. Re-running is safe: every table is overwritten.

All data is fictional. Marks, parties, people, amounts and outcomes are invented for the demo.

In [ ]:
BASE = "Files/ferrero_demo/csv"
TABLES = ["dim_mark", "dim_court", "dim_infringer", "dim_user", "fact_case", "fact_case_event", "fact_document",
          "fact_mailbox", "fact_intake_form", "fact_appstore", "fact_invoice", "golden_set"]
DATE_COLS = {"opened_date", "closed_date", "next_hearing_date", "next_action_due", "last_action_date", "event_date", "doc_date",
             "submitted_at", "detected_date", "takedown_requested_date", "removed_date", "invoice_date", "first_seen"}
TS_COLS = {"received_at", "triaged_at"}
DEMO_TODAY = "2026-11-05"   # frozen 'today' so the demo numbers never drift

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, TimestampType

def load(name):
    df = spark.read.option("header", True).option("inferSchema", True).option("multiLine", True).option("escape", '"').csv(f"{BASE}/{name}.csv")
    for c in df.columns:
        if c in DATE_COLS:
            df = df.withColumn(c, F.to_date(F.col(c).cast("string").substr(1, 10)))
        elif c in TS_COLS:
            df = df.withColumn(c, F.to_timestamp(F.col(c).cast("string")))
    return df

for name in TABLES:
    df = load(name)
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(name)
    print(f"{name:18s} {df.count():5d} rows, {len(df.columns):2d} columns")

## Date dimension and derived columns

In [ ]:
# dim_date: 2024-01-01 .. 2027-12-31, marked as the date table in the semantic model
dates = spark.sql("SELECT explode(sequence(to_date('2024-01-01'), to_date('2027-12-31'), interval 1 day)) AS Date")
dim_date = (dates
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
    .withColumn("YearQuarter", F.concat(F.year("Date"), F.lit(" Q"), F.quarter("Date")))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("Month", F.date_format("Date", "MMM yyyy"))
    .withColumn("MonthStart", F.trunc("Date", "month"))
    .withColumn("Week", F.weekofyear("Date"))
    .withColumn("WeekStart", F.date_sub("Date", (F.dayofweek("Date") + 5) % 7))
    .withColumn("DayName", F.date_format("Date", "EEE"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin([6, 7]))
    .withColumn("IsMonthEnd", F.col("Date") == F.last_day("Date")))
dim_date.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_date")
print("dim_date", dim_date.count())

In [ ]:
# fact_case: age buckets and a frozen 'as of' date for the demo
fc = spark.table("fact_case")
fc = (fc.withColumn("as_of_date", F.to_date(F.lit(DEMO_TODAY)))
        .withColumn("age_bucket", F.when(F.col("days_open") <= 90, "0–90 days").when(F.col("days_open") <= 180, "91–180 days")
                                   .when(F.col("days_open") <= 365, "181–365 days").otherwise("> 365 days"))
        .withColumn("is_overdue", (F.col("status") == "Open") & (F.col("next_action_due") < F.to_date(F.lit(DEMO_TODAY))))
        .withColumn("hearing_next_30d", (F.col("status") == "Open") & F.col("next_hearing_date").between(F.to_date(F.lit(DEMO_TODAY)), F.date_add(F.to_date(F.lit(DEMO_TODAY)), 30))))
fc.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("fact_case")

# fact_mailbox: a date key for dim_date and hours to triage
fm = spark.table("fact_mailbox")
fm = (fm.withColumn("received_date", F.to_date("received_at"))
        .withColumn("hours_to_triage", (F.unix_timestamp("triaged_at") - F.unix_timestamp("received_at")) / 3600.0))
fm.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("fact_mailbox")

# fact_appstore: days to takedown
fa = spark.table("fact_appstore").withColumn("days_to_takedown", F.datediff("removed_date", "takedown_requested_date"))
fa.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("fact_appstore")
print("derived columns written")

## Checks

In [ ]:
checks = {
  "events without a case":     "SELECT COUNT(*) FROM fact_case_event e LEFT ANTI JOIN fact_case c ON e.case_id = c.case_id",
  "documents without a case":  "SELECT COUNT(*) FROM fact_document d LEFT ANTI JOIN fact_case c ON d.case_id = c.case_id",
  "invoices without a case":   "SELECT COUNT(*) FROM fact_invoice i LEFT ANTI JOIN fact_case c ON i.case_id = c.case_id",
  "cases without a mark":      "SELECT COUNT(*) FROM fact_case c LEFT ANTI JOIN dim_mark m ON c.mark_id = m.mark_id",
  "cases without a court":     "SELECT COUNT(*) FROM fact_case c LEFT ANTI JOIN dim_court k ON c.court_id = k.court_id",
  "closed before opened":      "SELECT COUNT(*) FROM fact_case WHERE closed_date < opened_date",
}
for label, sql in checks.items():
    print(f"{label:28s} {spark.sql(sql).first()[0]}")
display(spark.sql("""SELECT status, COUNT(*) AS cases, SUM(claim_amount_aed) AS claimed_aed, SUM(recovered_amount_aed) AS recovered_aed
                     FROM fact_case GROUP BY status"""))

## Next steps (outside this notebook)

1. **Semantic model** – create a Direct Lake semantic model on this lakehouse with the tables above. Mark `dim_date` as the date table. Add the relationships and measures from the data pack (`Semantic_Model` and `DAX_Measures` sheets; also listed below).
2. **Row-level security** – create the roles below and map them to the Entra ID groups in `dim_user.entra_group`.
3. **Reports** – build the four dashboards from the `Dashboard_Specs` sheet (D1 Legal Case Overview, D2 Leadership Command View, D3 Infringement Intake and Triage, D4 Infringer and Enforcement Outcomes) and the drill-through pages DS-1 … DS-6.
4. **Documents** – upload `documents/` to the SharePoint libraries named in `fact_document.library_path`, so the Ferrero document sync and the golden set (`golden_set` table) line up with the dashboards.

### Relationships
| From | To | Cardinality | State |
|---|---|---|---|
| fact_case[mark_id] | dim_mark[mark_id] | Many-to-one | Active |
| fact_case[infringer_id] | dim_infringer[infringer_id] | Many-to-one | Active |
| fact_case[court_id] | dim_court[court_id] | Many-to-one | Active |
| fact_case[lead_counsel_id] | dim_user[user_id] | Many-to-one | Active |
| fact_case[opened_date] | dim_date[Date] | Many-to-one | Active |
| fact_case[closed_date] | dim_date[Date] | Many-to-one | Inactive (USERELATIONSHIP for closed-by-period) |
| fact_case_event[case_id] | fact_case[case_id] | Many-to-one | Active |
| fact_document[case_id] | fact_case[case_id] | Many-to-one | Active |
| fact_invoice[case_id] | fact_case[case_id] | Many-to-one | Active |
| fact_invoice[invoice_date] | dim_date[Date] | Many-to-one | Active |
| fact_mailbox[mark_id] | dim_mark[mark_id] | Many-to-one | Active |
| fact_mailbox[received_date] | dim_date[Date] | Many-to-one | Active (received_date = DATE of received_at) |
| fact_mailbox[linked_case_id] | fact_case[case_id] | Many-to-one | Inactive |
| fact_intake_form[mark_id] | dim_mark[mark_id] | Many-to-one | Active |
| fact_intake_form[submitted_at] | dim_date[Date] | Many-to-one | Active |
| fact_appstore[mark_id] | dim_mark[mark_id] | Many-to-one | Active |
| fact_appstore[detected_date] | dim_date[Date] | Many-to-one | Active |

### RLS roles
| Role | Table | Filter | Who |
|---|---|---|---|
| Legal-UAE | fact_case | `[country] = "UAE"` | IP and brand-protection team in the UAE |
| Legal-KSA | fact_case | `[country] = "KSA"` | Litigation counsel in KSA |
| Legal-MENA | fact_case | `NOT [country] IN {"UAE", "KSA"}` | Counsel covering Egypt, Jordan, Kuwait, Qatar, Oman, Turkey, India |
| Legal-Ops | (none) | `TRUE()` | Legal operations: all countries, all tables |
| Leadership | (none) | `TRUE()` | Head of Legal, Regional GC: all countries |
| Procurement | fact_document | `FALSE()  (object-level security: hide fact_document)` | Procurement sees cases and infringers, never documents |

### Measures
- **Cases** = `COUNTROWS(fact_case)`
- **Open Cases** = `CALCULATE([Cases], fact_case[status] = "Open")`
- **Closed Cases** = `CALCULATE([Cases], fact_case[status] = "Closed")`
- **New Cases** = `CALCULATE([Cases], USERELATIONSHIP(fact_case[opened_date], dim_date[Date]))`
- **New Cases 30d** = `CALCULATE([Cases], fact_case[opened_date] >= TODAY() - 30)`
- **Hearings Next 30d** = `CALCULATE([Cases], fact_case[status] = "Open", fact_case[next_hearing_date] >= TODAY(), fact_case[next_hearing_date] <= TODAY() + 30)`
- **Overdue Actions** = `CALCULATE([Cases], fact_case[status] = "Open", fact_case[next_action_due] < TODAY())`
- **Avg Days Open** = `CALCULATE(AVERAGEX(fact_case, DATEDIFF(fact_case[opened_date], TODAY(), DAY)), fact_case[status] = "Open")`
- **Avg Days to Close** = `CALCULATE(AVERAGEX(fact_case, DATEDIFF(fact_case[opened_date], fact_case[closed_date], DAY)), fact_case[status] = "Closed")`
- **Open Cases at Date** = `VAR d = MAX(dim_date[Date]) RETURN CALCULATE(COUNTROWS(fact_case), REMOVEFILTERS(dim_date), fact_case[opened_date] <= d, OR(ISBLANK(fact_case[closed_date]), fact_case[closed_date] > d))`
- **Claimed** = `SUM(fact_case[claim_amount_aed])`
- **Awarded** = `SUM(fact_case[awarded_amount_aed])`
- **Settled** = `SUM(fact_case[settlement_amount_aed])`
- **Awarded or Settled** = `[Awarded] + [Settled]`
- **Recovered** = `SUM(fact_case[recovered_amount_aed])`
- **Recovery Rate** = `DIVIDE([Recovered], [Awarded or Settled])`
- **Decided Cases** = `CALCULATE([Cases], fact_case[outcome] IN {"Won", "Partially won", "Settled", "Lost", "Takedown achieved"})`
- **Win Rate** = `DIVIDE(CALCULATE([Cases], fact_case[outcome] IN {"Won", "Partially won", "Settled", "Takedown achieved"}), [Decided Cases])`
- **Legal Spend** = `SUM(fact_invoice[amount_aed])`
- **Spend to Recovery** = `DIVIDE([Legal Spend], [Recovered])`
- **Reports** = `COUNTROWS(fact_mailbox)`
- **Reports 30d** = `CALCULATE([Reports], fact_mailbox[received_at] >= TODAY() - 30)`
- **Backlog** = `CALCULATE([Reports], fact_mailbox[triage_status] = "New")`
- **SLA Met %** = `1 - DIVIDE(CALCULATE([Reports], fact_mailbox[sla_breached] = TRUE()), CALCULATE([Reports], NOT ISBLANK(fact_mailbox[triaged_at])))`
- **Median Hours to Triage** = `MEDIANX(FILTER(fact_mailbox, NOT ISBLANK(fact_mailbox[triaged_at])), DATEDIFF(fact_mailbox[received_at], fact_mailbox[triaged_at], HOUR))`
- **Conversion to Case %** = `DIVIDE(CALCULATE([Reports], fact_mailbox[triage_status] = "Linked to case"), [Reports])`
- **Takedowns Completed** = `CALCULATE(COUNTROWS(fact_appstore), fact_appstore[status] = "Removed")`
- **Takedown Success %** = `DIVIDE([Takedowns Completed], CALCULATE(COUNTROWS(fact_appstore), fact_appstore[status] IN {"Removed", "Rejected by store", "Re-listed", "Takedown requested"}))`
- **Avg Days to Takedown** = `AVERAGEX(FILTER(fact_appstore, NOT ISBLANK(fact_appstore[removed_date])), DATEDIFF(fact_appstore[takedown_requested_date], fact_appstore[removed_date], DAY))`
- **Active Infringers** = `CALCULATE(DISTINCTCOUNT(fact_case[infringer_id]), fact_case[status] = "Open")`
- **Repeat Offenders** = `CALCULATE(DISTINCTCOUNT(dim_infringer[infringer_id]), dim_infringer[repeat_offender] = TRUE())`
- **Documents Indexed %** = `DIVIDE(CALCULATE(COUNTROWS(fact_document), fact_document[index_status] IN {"Indexed", "Indexed (flagged)"}), COUNTROWS(fact_document))`